Business value: how accuracy translates into recommendation outcomes. Source: `modeling/03_compare.py` → `modeling/out/results.json` (2023 test year, slots per quarter = number of true hits, about 203).

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
load = lambda rel: json.loads((ROOT / rel).read_text())
show = lambda rel, width=720: display(Image(filename=str(ROOT / rel), width=width))
pd.set_option("display.precision", 3)

In [2]:
imp = load("modeling/out/results.json")["impact"]
print(f"{imp['quarters']} quarters, about {imp['books_per_quarter']:,} new serials and {imp['slots_per_quarter']} slots per quarter")
pd.DataFrame({"future hits per quarter": imp["hits_per_quarter"], "eventual followers of the picks": imp["eventual_followers_of_picks_per_quarter"]}).rename(
    index={"rule_hot_genre": "rule: popular genres first", "lr_SAT": "main model", "oracle": "perfect foresight"})

4 quarters, about 2,019 new serials and 203 slots per quarter


,future hits per quarter,eventual followers of the picks
random,20.3,42081
rule: popular genres first,53.0,118459
main model,97.8,237615
perfect foresight,NaN,372252


Eventual followers are an outcome, not a causal effect: the model's picks would have been popular anyway. Turning picks into revenue needs an assumed lift $\lambda$ from a slot and a value per view:

$$
\Delta\text{Revenue per year}=\sum_{q=1}^{4}\Big(\sum_{i\in S_q^{\text{model}}}V_i-\sum_{i\in S_q^{\text{rule}}}V_i\Big)\cdot\lambda\cdot\frac{\text{CPM}}{1000}
$$

$\lambda$ is unknown from this data, which is why the pilot should be an A/B test.

How many slots: more slots find more hits but each extra slot is less likely to be a hit. Keep adding slots while the next serial's hit probability times the value of a hit exceeds the cost of a slot:

$$
\text{Value}(K)=b\cdot\sum_{k=1}^{K}p_{(k)}-a\cdot K \quad\Rightarrow\quad \text{stop when } p_{(K)}\approx a/b
$$

Pending: the slides use a fixed 50 slots per quarter (53% → 68% of slots, 34 vs 14 hits per quarter), a day-30 re-rank (68% → 76%), and revenue and editor-time estimates (about $39k and $36k a year). The code for these is not in the repo yet; this notebook will read its output once it is added.

Takeaways: with about 203 slots a quarter the model fills 98 with future hits, against 53 for the best rule and 20 at random. The money value depends on assumptions (slot lift, ad revenue per view) that an A/B pilot should measure.